<a id="top"></a>
# House Sales: business & data understanding (esercitazione)

In questa esercitazione si lavora su un caso immobiliare basato su dati reali di compravendita di case nella King County (area di Seattle, USA), raccolti in circa un anno di osservazione. L’obiettivo è allenarsi a leggere la richiesta del business e a collegarla ai dati disponibili.

<a id="indice"></a>
## Indice

- [Contesto e obiettivo del progetto](#contesto-obiettivo)
- [Richieste del business](#richieste-business)
- [Da dove arrivano i dati](#origine-dati)
- [Caricamento del dataset House Sales](#caricamento-dataset)
- [Primo sguardo ai dati](#primo-sguardo)
- [Prime esplorazioni guidate](#prime-esplorazioni)
- [Punti chiave da portare avanti](#sintesi)


<a id="contesto-obiettivo"></a>
## Contesto e obiettivo del progetto

Immagina di lavorare nel team dati di una grande società immobiliare attiva nella contea di King. L’azienda compra e rivende case e offre anche un servizio di consulenza ai privati su come prezzare correttamente il proprio immobile.

Negli ultimi anni il management ha iniziato a percepire alcuni problemi ricorrenti:

- alcune case restano sul mercato per mesi perché il prezzo iniziale è troppo alto;
- altre vengono vendute velocemente, ma con la sensazione di aver “lasciato soldi sul tavolo”;
- in certe zone il mercato sembra più sensibile alla metratura, in altre alla qualità della casa o alla vicinanza a certe aree.

La direzione commerciale chiede al team dati di costruire uno strumento che aiuti a:

- stimare un intervallo di prezzo realistico per una casa prima di metterla sul mercato;
- capire quali caratteristiche spingono di più il prezzo verso l’alto o verso il basso;
- evidenziare casi potenzialmente sottostimati o sovrastimati rispetto al mercato della zona.

Dal punto di vista tecnico possiamo formulare il problema come un task di **regressione**: dato un insieme di caratteristiche di una casa, vogliamo prevedere il suo **prezzo di vendita**.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Scrivi in poche righe (fuori da questo notebook) come presenteresti il problema a una direttrice commerciale che non conosce il machine learning. Prova a usare parole semplici e a distinguere:

- che cosa vuole ottenere il business;
- che cosa pensi di poter consegnare come output del progetto (es. stime, report, segmentazioni).

</div>


<a id="richieste-business"></a>
## Richieste del business

Per lavorare bene serve esplicitare le richieste, anche quando arrivano in modo un po’ informale. In questo caso possiamo riassumerle in alcune domande guida:

1. **Prezzo atteso per una nuova casa**  
   Dato un immobile con certe caratteristiche (metratura, numero di stanze, zona, condizioni), quale fascia di prezzo è plausibile?

2. **Fattori che guidano il prezzo**  
   Quanto pesano davvero le diverse caratteristiche sul prezzo finale? Ad esempio: contano di più i metri quadri, la posizione o lo stato dell’immobile?

3. **Differenze tra quartieri / zip code**  
   A parità di caratteristiche, ci sono zone in cui i prezzi sono sistematicamente più alti o più bassi?

4. **Casi anomali da attenzionare**  
   Ci sono case che sembrano molto più costose o molto più economiche rispetto a case simili?

Nel resto dell’esercitazione userai il dataset House Sales per capire **se** e **in che misura** queste domande possono trovare risposta con i dati a disposizione.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Rileggi le quattro domande sopra e, per ciascuna, prova a decidere se ti sembra:

- principalmente una domanda di **previsione** (voglio un numero),
- una domanda di **spiegazione** (voglio capire cosa succede),
- oppure una combinazione delle due.

Annota le tue scelte: torneranno utili quando inizierai a pensare ai modelli.

</div>


<a id="origine-dati"></a>
## Da dove arrivano i dati

Il dataset `house_sales` raccoglie informazioni su migliaia di compravendite di case nella **King County** (l’area di Seattle, nello stato di Washington, USA). I dati coprono circa un anno di transazioni, tra il 2014 e il 2015.

Ogni riga rappresenta una casa venduta; le colonne descrivono:

- il **prezzo di vendita** (`price`);
- caratteristiche fisiche dell’immobile, come:
  - `bedrooms`, `bathrooms`;
  - `sqft_living`, `sqft_lot` (metratura interna e del lotto);
  - `floors`, `sqft_above`, `sqft_basement`;
- indici qualitativi come `condition` e `grade`, che riassumono la qualità percepita dell’abitazione;
- informazioni sulla **posizione**:
  - `zipcode` (zona),
  - `lat`, `long` (coordinate geografiche);
- alcune variabili di contesto, ad esempio:
  - la **data di vendita** (`date`);
  - misure di metratura della casa e del lotto nelle vicinanze (`sqft_living15`, `sqft_lot15`).

Tutti questi elementi sono potenziali candidati come feature per il modello, ma in questa fase li guardiamo ancora solo come **pezzi di informazione** a disposizione.

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Attenzione**

Il dataset rappresenta il mercato immobiliare **in una singola area geografica** e in un **periodo di tempo limitato**. Le relazioni che troverai (per esempio tra metratura e prezzo) non vanno automaticamente estese ad altre città o ad altri anni.

</div>


<a id="caricamento-dataset"></a>
## Caricamento del dataset House Sales

Per lavorare con i dati useremo `pandas` come libreria principale. In questo esercizio il dataset viene scaricato da OpenML tramite `scikit-learn`, così non serve gestire file manualmente.

- `fetch_openml` scarica il dataset da OpenML e lo restituisce come oggetto simile a un dizionario con dati e metadati.  
  Documentazione: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.fetch_openml.html
- `pandas.DataFrame` è la struttura tabellare base di pandas.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html

Quando si usa `as_frame=True`, l’oggetto restituito da `fetch_openml` contiene un attributo `.frame` che è un `DataFrame` con tutte le colonne (feature e target) già unite.

L’obiettivo di questa sezione è:

- ottenere un `DataFrame` con tutte le colonne (feature + target);
- verificare rapidamente quante righe e quante colonne ci sono.


In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml  
import plotly.express as px

# Scarica il dataset House Sales da OpenML
raw = fetch_openml(data_id=42092, as_frame=True)  
# Estrai il DataFrame con tutte le colonne
house = raw.frame.copy()  

# Controlla dimensioni: numero di righe e colonne
house.shape


<a id="primo-sguardo"></a>
## Primo sguardo ai dati

Una volta ottenuto il `DataFrame`, il passo successivo è capire che forma hanno i dati:

- quali colonne ci sono;
- che tipo di valori contengono;
- se ci sono dati mancanti significativi.

Per farlo useremo alcuni metodi fondamentali di `pandas`:

- `DataFrame.head(n)` mostra le prime `n` righe (di default 5).  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html
- `DataFrame.info()` riassume le colonne, i tipi di dato e il numero di valori non nulli.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.info.html
- `DataFrame.describe()` calcola statistiche descrittive per le colonne numeriche (media, deviazione standard, quantili…).  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html
- `DataFrame.isna()` restituisce una maschera booleana con `True` dove il valore è mancante.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Completa gli `___` nel blocco di codice successivo per:

1. visualizzare le prime righe del dataset;
2. ottenere un riepilogo delle colonne;
3. stimare la percentuale di valori mancanti per colonna.

</div>


In [ ]:
# 1. Prime righe
house_head = house.___(10)  # TODO: metodo per mostrare le prime righe
house_head

# 2. Schema delle colonne
house_info = house.___()    # TODO: metodo per riassumere dtypes e valori non nulli

# 3. Percentuale di valori mancanti per colonna
missing_ratio = house.isna().mean().sort_values(ascending=False)
missing_ratio

# Domanda: perché, in questa fase, è utile guardare sia lo schema (info) sia i valori mancanti?
# Scrivi una risposta breve come commento o in un tuo appunto.


<a id="prime-esplorazioni"></a>
## Prime esplorazioni guidate

Per collegare meglio le domande del business ai dati, partiamo da due esplorazioni semplici:

1. la distribuzione del **prezzo di vendita**;
2. la relazione tra **numero di camere da letto** e prezzo medio.

Useremo:

- `plotly.express.histogram` per il grafico di distribuzione.  
  Documentazione: https://plotly.com/python-api-reference/generated/plotly.express.histogram.html
- `DataFrame.groupby()` per raggruppare le case per numero di camere da letto.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html
- `Series.mean()` per calcolare il prezzo medio per ciascun gruppo.  
  Documentazione: https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html
- `plotly.express.bar` per visualizzare il prezzo medio per numero di camere.  
  Documentazione: https://plotly.com/python-api-reference/generated/plotly.express.bar.html

<div style="background-color:#f5f5f5; padding:10px; border-left:4px solid #999;">

**Esercizio**

Nel blocco di codice seguente:

1. completa gli `___` per:
   - specificare la colonna del prezzo;
   - completare il `groupby`;
   - indicare la colonna da usare sull’asse `x` e sull’asse `y`;
2. osserva i due grafici e prova a rispondere:
   - la distribuzione dei prezzi è simmetrica o molto asimmetrica?
   - il prezzo medio cresce sempre con il numero di camere?

Scrivi le tue osservazioni come commento nel codice o in un tuo appunto.

</div>


In [ ]:
# Distribuzione dei prezzi di vendita
fig_price = px.histogram(
    house,
    x="___",          # TODO: nome della colonna del prezzo
    nbins=50,
    title="Distribuzione dei prezzi di vendita (House Sales)"
)
fig_price.show()

# Prezzo medio per numero di camere da letto
price_by_bedrooms = (
    house
    .groupby("___")["___"]   # TODO: colonna per raggruppare, colonna del prezzo
    .mean()
    .reset_index()
    .sort_values("bedrooms")
)

fig_bedrooms = px.bar(
    price_by_bedrooms,
    x="___",   # TODO: numero di camere
    y="___",   # TODO: prezzo medio
    title="Prezzo medio per numero di camere da letto"
)
fig_bedrooms.show()


<a id="sintesi"></a>
## Punti chiave da portare avanti

Prima di passare alla modellazione, conviene fissare alcuni elementi essenziali emersi da questa esercitazione:

- il problema di business è naturalmente formulabile come **previsione di un prezzo continuo** (task di regressione);
- il dataset contiene migliaia di transazioni reali, con un insieme ricco di feature sulla casa e sulla posizione;
- la distribuzione dei prezzi tende a essere molto asimmetrica, con una coda di case molto costose;
- alcune variabili, come il numero di camere o la metratura interna, mostrano da subito una relazione chiara con il prezzo.

Nel proseguimento del corso lo scopo sarà usare queste osservazioni per:

- pulire e preparare i dati in modo coerente con il problema;
- progettare un modello che dia risposte utili alle domande del business, non solo buone metriche tecniche.
